In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [ ]:
df = pd.read_csv("/kaggle/input/System-Threat-Forecaster/train.csv")  # Train 76, target
X_test = pd.read_csv("/kaggle/input/System-Threat-Forecaster/test.csv")  # X_Test 75
print(df.shape, X_test.shape)

# EDA

Missing values, only 1 value, feature engineering, num and cat columns, duplicates, outliers, reduce cardinality

In [ ]:
missing_percentage = df.isna().mean() * 100

columns_above_4_percent = missing_percentage[missing_percentage >= 4 ].index.tolist()

print("Columns with more than 4% missing values:", columns_above_4_percent)

In [ ]:
# Filter columns with only 1 unique value

unique_cols = df.nunique() 
single_value_cols = unique_cols[unique_cols == 1].index.tolist()  
print("Columns with only one unique value:", single_value_cols)

unique_cols = X_test.nunique()  # Count unique values per column
single_value_cols = unique_cols[unique_cols == 1].index.tolist()  
print("Columns with only one unique value:", single_value_cols)

print(df["IsBetaUser"].unique(),
df['AutoSampleSubmissionEnabled'].unique(),
df['IsFlightsDisabled'].unique(),

X_test["IsBetaUser"].unique(),
X_test['AutoSampleSubmissionEnabled'].unique(),
X_test['IsFlightsDisabled'].unique())

In [ ]:

df['DateAS'] = pd.to_datetime(df['DateAS'], errors='coerce')
df['DateOS'] = pd.to_datetime(df['DateOS'], errors='coerce')

df['DateAS'].fillna(df['DateAS'].min(), inplace=True)
df['DateOS'].fillna(df['DateOS'].min(), inplace=True)

df['Days_Between_os_as'] = (df['DateOS'] - df['DateAS']).dt.days

df['DateAS_Year'] = df['DateAS'].dt.year
df['DateAS_Month'] = df['DateAS'].dt.month
df['DateAS_Day'] = df['DateAS'].dt.day

df['DateOS_Year'] = df['DateOS'].dt.year
df['DateOS_Month'] = df['DateOS'].dt.month
df['DateOS_Day'] = df['DateOS'].dt.day

current_date = pd.Timestamp.now()
df['DaysSinceASUpdate'] = (current_date - df['DateAS']).dt.days
df['DaysSinceOSUpdate'] = (current_date - df['DateOS']).dt.days

df = df.drop(['DateAS', 'DateOS', "MachineID",'IsBetaUser', 'AutoSampleSubmissionEnabled', 'IsFlightsDisabled'], axis=1)

# same preprocessing to the X_test
X_test['DateAS'] = pd.to_datetime(X_test['DateAS'], errors='coerce')
X_test['DateOS'] = pd.to_datetime(X_test['DateOS'], errors='coerce')
X_test['Days_Between_os_as'] = (X_test['DateOS'] - X_test['DateAS']).dt.days
X_test['DateAS_Year'] = X_test['DateAS'].dt.year
X_test['DateAS_Month'] = X_test['DateAS'].dt.month
X_test['DateAS_Day'] = X_test['DateAS'].dt.day
X_test['DateOS_Year'] = X_test['DateOS'].dt.year
X_test['DateOS_Month'] = X_test['DateOS'].dt.month
X_test['DateOS_Day'] = X_test['DateOS'].dt.day
X_test['DaysSinceASUpdate'] = (current_date - X_test['DateAS']).dt.days
X_test['DaysSinceOSUpdate'] = (current_date - X_test['DateOS']).dt.days
X_test = X_test.drop(['DateAS', 'DateOS',"MachineID",'IsBetaUser', 'AutoSampleSubmissionEnabled', 'IsFlightsDisabled'], axis=1)

In [ ]:
cat_X = df.select_dtypes(exclude=['number', "float"])  #strg
is_col = [col for col in df.columns if col.startswith('Is') or col.startswith('Has')]  
bin_col = [col for col in df.columns if df[col].nunique() == 2]
cat_col = list(set (list(cat_X.columns) + list(set(is_col)& set(bin_col))) )

num_X = df.select_dtypes(include=["number", "float"])
num_col = [col for col in num_X.columns if col not in cat_col]
num_col.remove("target")

print("Categorical columns:", len(cat_col),cat_col) #36
print("Numeric columns:", len(num_col),num_col) #42

# print(df[cat_col].isna().sum())
# print(df[num_col].isna().sum())


one_hot_cols = ['HasTpm', 'IsSecureBootEnabled', 'IsPenCapable', 'HasOpticalDiskDrive',
                'IsPassiveModeEnabled', 'IsSystemProtected', 'IsAlwaysOnAlwaysConnectedCapable',
                'IsPortableOS', 'IsTouchEnabled', 'IsVirtualDevice', 'IsGamer']

ordinal_cols = ['OSVersion', 'Processor', 'OSArchitecture', 'LicenseActivationChannel',
                'OSEdition', 'OSInstallType', 'SKUEditionName', 'OSSkuFriendlyName', 
                'PowerPlatformRole']

frequency_cols = ['ProductName', 'OSBuildLab', 'AppVersion', 'EngineVersion', 
                  'NumericOSVersion', 'SignatureVersion', 'OSBranch', 
                  'OSGenuineState', 'AutoUpdateOptionsName', 'OsPlatformSubRelease']

target_cols = ['ChassisType', 'PlatformType', 'PrimaryDiskType', 'MDC2FormFactor',
               'OSInstallType', 'DeviceFamily', 'FlightRing']

In [ ]:
X = df.drop(columns=["target"])
y = df['target']

print(X.columns)

In [ ]:
print((X.isna().sum().sum()))
print((X_test.isna().sum().sum()))

In [ ]:
X_strip = X.map(lambda x: x.strip() if isinstance(x, str) else x)
X_test = X_test.map(lambda x: x.strip() if isinstance(x, str) else x)

#print("Duplicates in X_test before removal:", Xtest_strip.duplicated().sum()) #DONOT remove

X_no_duplicates = X_strip.drop_duplicates()
y_no_duplicates = y.loc[X_no_duplicates.index]

print("Shape of X before:", X.shape, "After:", X_no_duplicates.shape)
print("Shape of y before:", y.shape, "After:", y_no_duplicates.shape)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy_model = DummyClassifier(strategy="most_frequent").fit(X_no_duplicates,y_no_duplicates)
y_pred=dummy_model.predict(X_test) #y_test not given

# submission = pd.DataFrame({"id": range(0,len(y_pred)),
#                            "target": y_pred})
# submission.to_csv('submission.csv',index=False) 
# print(submission) 

# print (accuracy_score(y_val,dummy_model.predict(X_val))) #0.5016

In [ ]:
from scipy import stats

# Function to count outliers using Z-score
def count_outliers_zscore(df, threshold=10): #to identify extreme values
    outlier_counts = {}
    
    for col in df.select_dtypes(include=[np.number]):  # Select only numerical columns
        z_scores = np.abs(stats.zscore(df[col])) #(x-mean)/sd
        outliers = df[col][z_scores > threshold]
        
        outlier_counts[col] = len(outliers)
    
    return outlier_counts

outlier_counts_df = count_outliers_zscore(df)
print("Outliers per column (Z-score):", outlier_counts_df)
print("Total df outliers using z score:", sum(outlier_counts_df.values()))

outlier_counts_test = count_outliers_zscore(X_test)
print("Total X_test outliers using z score:", sum(outlier_counts_test.values()))


def cap_outliers(df, lower_percentile=0.01, upper_percentile=0.99): #0.01 to 0.99
    df_capped = df.copy()
    
    for col in df_capped.select_dtypes(include=[np.number]):
        lower_limit, upper_limit = df_capped[col].quantile([lower_percentile, upper_percentile])
        df_capped[col] = np.clip(df_capped[col], lower_limit, upper_limit) #caps the values of a column
    
    return df_capped

df_capped = cap_outliers(df)
X_no_duplicates_capped = cap_outliers(X_no_duplicates) #without outliers
X_test_capped = cap_outliers(X_test) #without outliers

outlier_counts_df = count_outliers_zscore(df_capped)
print("Total df cap outliers:", sum(outlier_counts_df.values()))

outlier_counts_X_no_duplicates_capped = count_outliers_zscore(X_no_duplicates_capped)
print("Total X outliers:", sum(outlier_counts_X_no_duplicates_capped.values()))

outlier_counts_test = count_outliers_zscore(X_test_capped)
print("Total X_test outliers:", sum(outlier_counts_test.values()))


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from collections import Counter
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer

# grouping rare categories into a single category "Others".
def reduce_cardinality(X, threshold=0.01):
    # Convert X to a DataFrame if it's not already one
    
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    
    X_reduced_car = X.copy()
    for column in X_reduced_car.columns:
        X_reduced_car[column] = X_reduced_car[column].astype(str) 
        value_counts = X_reduced_car[column].value_counts(normalize=True) #proportion
        frequent_categories = value_counts[value_counts >= threshold].index
        X_reduced_car[column] = np.where(X_reduced_car[column].isin(frequent_categories), X_reduced_car[column], 'Others')
    
    return X_reduced_car.to_numpy()


**Visualization**

In [ ]:
import matplotlib.pyplot as plt
print(df.shape)
class_counts = df['target'].value_counts()
print("Class Distribution:\n", class_counts)

# Plot
class_counts.plot(kind='bar', title='Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

# Imbalance Ratio 1/0
imbalance_ratio = class_counts.min() / class_counts.max()
print(f"Imbalance Ratio: {imbalance_ratio:.2f}")

# If ratio < 0.5, data is imbalanced
if imbalance_ratio < 0.5:
    print("Data is imbalanced.")
else:
    print("Data is balanced.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#correlation more than threshold
corr_matrix = df[num_col].corr()
mask = (corr_matrix < 0.65) & (corr_matrix > -0.65)
filtered_corr = corr_matrix.mask(mask) 
plt.figure(figsize=(10, 6))
sns.heatmap(filtered_corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Correlation Heatmap (|corr| > 0.65)")
plt.show()


In [ ]:
from scipy.stats import zscore
import seaborn as sns

z_scores = np.abs(df[num_col].apply(zscore))
outlier_mask = z_scores > 2  
outlier_counts = outlier_mask.sum()
threshold = 0.1 * len(df)  # 10% of rows
high_outlier_cols = outlier_counts[outlier_counts > threshold].index

df_high_outliers = df[high_outlier_cols]

if not df_high_outliers.empty:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df_high_outliers, palette="coolwarm")
    plt.xticks(rotation=90)
    plt.title(f"Columns with Many Outliers (>10% of data, Z > 2)") #more than 2 standard deviations from the mean.
    plt.show()
else:
    print("No columns have more than 10% outliers.")
    
def plot_top_outlier_boxplots(df, lower_percentile=0.01, upper_percentile=0.99, top_n=10):
    outlier_ratios = {}  
    for col in df[num_col]:  
        lower_limit, upper_limit = df[col].quantile([lower_percentile, upper_percentile])

        # Count how many values are outliers
        outlier_count = ((df[col] < lower_limit) | (df[col] > upper_limit)).sum()
        outlier_ratio = outlier_count / len(df)

        if outlier_ratio > 0:  # Store only columns with outliers
            outlier_ratios[col] = outlier_ratio

    # Select top N columns with the most outliers
    top_outlier_cols = sorted(outlier_ratios, key=outlier_ratios.get, reverse=True)[:top_n]

    # Plot boxplots only for top outlier columns
    if top_outlier_cols:
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=df[top_outlier_cols], palette="coolwarm")
        plt.xticks(rotation=90)
        plt.title(f"Top {top_n} Columns with Most Outliers (<1% or >99%)")
        plt.show()
    else:
        print("No significant outliers found.")

plot_top_outlier_boxplots(df)

# Pipeline

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, RobustScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer , KNNImputer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="mean")),  #knn, itearive not good
    ('scaler', StandardScaler())   #Robust not good
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  
    ('cardinality_reducer', FunctionTransformer(reduce_cardinality, kw_args={'threshold': 0.01})),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))   #OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_col),
        ('cat', categorical_transformer, cat_col)
    ]
) 
#only takes x as param

# Build the preprocessing pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

pipeline.fit(X_no_duplicates, y_no_duplicates)

In [ ]:
X_test_pre =pipeline.transform(X_test_capped)

# X_test_pre_dense = X_test_pre.toarray()
print(type(X_test_pre), X_test_pre.dtype) 

X_test_pre_df=pd.DataFrame(X_test_pre)

print(X_test.isna().sum().sum())  
print(X_test_pre_df.isna().sum().sum())  #0

print(X.shape)
print(X_test.shape)
print(X_test_pre.shape)
print(X_test_pre_df.shape)


In [ ]:
X_pre = pipeline.transform(X_no_duplicates_capped)
# X_pre_dense = X_pre.toarray()
X_pre_df=pd.DataFrame(X_pre) 

print(X_pre.shape)
print(X_pre_df.isna().sum().sum()) #0

# Split

In [ ]:
from sklearn.model_selection import train_test_split # for train - val split

X_train, X_val, y_train, y_val = train_test_split(X_pre_df, y_no_duplicates , test_size=0.2, random_state=1)

print("Training Set Shape:", X_train.shape)
print("Validation Set Shape:", X_val.shape)


In [ ]:
print(X_train.head(2), "end",
      X_val.head(2),"y _val",
     y_val.head(2))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score

linear_model = LinearRegression() #convex
linear_model.fit(X_train, y_train) #after pipeline

y_val_pred_linear = linear_model.predict(X_val)

In [ ]:
y_val_pred_linear_binary = (y_val_pred_linear >= 0.531).astype(int) #0.604, 0.6098

score_lin = accuracy_score(y_val, y_val_pred_linear_binary)
print("linear reg benchmark for X_val", score_lin) 

# y_pred= linear_model.predict(X_test_pre_df)
# y_pred_binary = (y_pred >= 0.5).astype(int)
# y_pred_binary


In [ ]:
import matplotlib.pyplot as plt    
b = plt.hist(y_val_pred_linear , bins=50)

# Features Selection

In [ ]:
from sklearn.decomposition import PCA #dimensionality reduction technique
#unsuper, find patterns in X

pca = PCA(n_components=0.95, random_state=1)
pca.fit(X_train)

X_train_pca = pca.transform(X_train)
X_val_pca = pca.transform(X_val)
X_test_pca= pca.transform(X_test_pre_df)

print(f"Explained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"Number of components: {pca.n_components_}") #69, 75, 55 , robust 23, many encod 36

#X_pca_dense = X_train_pca.toarray()
X_pca_df=pd.DataFrame(X_train_pca)
X_pca_df
X_val_pca_df=pd.DataFrame(X_val_pca)
X_val_pca_df
X_test_pca_df=pd.DataFrame(X_test_pca)
X_test_pca_df


In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_pca_df, y_train)

y_val_pred = linear_model.predict(X_val_pca_df)

print(np.unique(y_val_pred)) #yes unique

In [ ]:
y_val_pred_binary = (y_val_pred >= 0.51).astype(int)
score_lin = accuracy_score(y_val, y_val_pred_binary)
print("linear reg score on PCA", score_lin) # 0.6033 -0.51

# y_pred= linear_model.predict(X_test_pre_df)
# y_pred_binary = (y_pred >= 0.51).astype(int)
# y_pred_binary

In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score

# rf = RandomForestClassifier(
#     n_estimators=200,   # More trees = better generalization
#     max_depth=20,       # Deeper trees
#     min_samples_split=2,
#     min_samples_leaf=1,
#     random_state=42,
#     n_jobs=-1
# )

# rf.fit(X_pca_df, y_train)
# y_pred = rf.predict(X_val_pca_df)
# print("Random Forest Accuracy on PCA:", accuracy_score(y_val, y_pred)) #0.6051

# #--

# rf.fit(X_train, y_train)
# y_pred = rf.predict(X_val)
# print("Random Forest Accuracy on X_val:", accuracy_score(y_val, y_pred)) #0.6199


# Model on X Train

In [ ]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier


In [ ]:

# classification_models = {
#     "Logistic Regression": LogisticRegression(),
#     "Ridge Classifier": RidgeClassifier(),
#     "Decision Tree": DecisionTreeClassifier(max_depth=5),
#     "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5), #bagging->less over, ->less var 
#     "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=5),
#     "SVC": SVC(kernel='rbf'),
#     "Gaussian Naive Bayes": GaussianNB(),
#     "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
#     "XGBoost": XGBClassifier(n_estimators=100, max_depth=5), 
#     "LightGBM": LGBMClassifier(n_estimators=100, max_depth=5),
#     "CatBoost": CatBoostClassifier(iterations=100, depth=5, verbose=False),
#     "AdaBoost": AdaBoostClassifier(n_estimators=100),
#     "SGD Classifier": SGDClassifier(max_iter=1000, tol=1e-3),
#     "MLP Classifier": MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000)
# }


In [ ]:
# results = {}

# for name, model in classification_models.items():

#     model.fit(X_pca_df, y_train)
#     y_val_pred = model.predict(X_val_pca_df)a
#     acc = accuracy_score(y_val, y_val_pred)
#     results[name] = acc

# for name, acc in results.items():
#     print(f"{name}: Accuracy = {acc:.4f}")

# # RESULT
# # Logistic Regression: Accuracy = 0.6046
# # Ridge Classifier: Accuracy = 0.6044
# # Decision Tree: Accuracy = 0.5696
# # Random Forest: Accuracy = 0.5905
# # Gradient Boosting: Accuracy = 0.6053
# # SVC: Accuracy = 0.6146                sub  #0.6017 poly, 0.52 sigmoid
# # Gaussian Naive Bayes: Accuracy = 0.5543
# # K-Nearest Neighbors: Accuracy = 0.5579
# # XGBoost: Accuracy = 0.5998             sub
# # LightGBM: Accuracy = 0.6080            sub 
# # CatBoost: Accuracy = 0.6009
# # AdaBoost: Accuracy = 0.5988 sub
# # SGD Classifier: Accuracy = 0.5917
# # MLP Classifier: Accuracy = 0.5823        sub


In [ ]:
# log_reg_2 = LogisticRegression(C=1, penalty='l1', solver='liblinear', max_iter=1000, class_weight='balanced')
# log_reg_2.fit(X_train, y_train)
# print(log_reg_2.score(X_val,y_val)) #0.6032 pca, 0.6112 train


In [ ]:
# from sklearn.ensemble import GradientBoostingClassifier #slower than lgbm,xtreme

# gb = GradientBoostingClassifier(random_state=42)

# param_grid_gb = {
#     'n_estimators': [50, 100],
#     'learning_rate': [0.1, 0.01],
#     'max_depth': [3, 5]
# }

# grid_search_gb = GridSearchCV(gb, param_grid_gb, cv=3, scoring='accuracy', n_jobs=-1)
# grid_search_gb.fit(X_pca_df, y_train)

# best_gb = grid_search_gb.best_estimator_
# gb_val_accuracy = best_gb.score(X_val_pca_df, y_val)

# print(f"Best Parameters for Gradient Boosting: {grid_search_gb.best_params_}")
# print(f"Validation Accuracy: {gb_val_accuracy:.4f}")
# # Best Parameters for Gradient Boosting: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
# # Validation Accuracy: 0.6240
# ## Test 0.62740


In [ ]:

from lightgbm import LGBMClassifier #Light Gradient Boosting Machine (fast) histomgram bining

lgb = LGBMClassifier(random_state=42)

param_grid_lgb = {
    'n_estimators': [50, 100],
    'learning_rate': [0.1, 0.01],
    'max_depth': [3, 5, 7]
}

grid_search_lgb = GridSearchCV(lgb, param_grid_lgb, cv=3, scoring='accuracy', n_jobs=1)
grid_search_lgb.fit(X_train, y_train)

best_lgb = grid_search_lgb.best_estimator_
lgb_val_accuracy = best_lgb.score(X_val, y_val)

print(f"Best Parameters for LightGBM: {grid_search_lgb.best_params_}")
print(f"Validation Accuracy: {lgb_val_accuracy:.4f}")
# Best Parameters for LightGBM: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 100}
# Validation Accuracy: 0.6253
# test 0.6327


In [ ]:
from xgboost import XGBClassifier

clf1 = XGBClassifier(random_state=1 , max_depth = 100)
clf1.fit(X_train,y_train)

score = clf1.score(X_val, y_val)
print(f"Model score: {score}") #0.614 train submitted

In [ ]:
from xgboost import XGBClassifier #Extreme Gradient Boosting
from sklearn.model_selection import GridSearchCV


xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric="logloss")

param_grid_xgb = {
    'n_estimators': [50, 100],
    'learning_rate': [0.1, 0.01],
    'max_depth': [3, 5, 7]
}

grid_search_xgb = GridSearchCV(xgb, param_grid_xgb, cv=3, scoring='accuracy', n_jobs=1)
grid_search_xgb.fit(X_train, y_train)

best_xgb = grid_search_xgb.best_estimator_
xgb_val_accuracy = best_xgb.score(X_val, y_val)

print(f"Best Parameters for XGBoost: {grid_search_xgb.best_params_}")
print(f"Validation Accuracy: {xgb_val_accuracy:.4f}")

# Best Parameters for XGBoost: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
# Validation Accuracy: 0.6270  
# test 0.628


In [ ]:
# import numpy as np

# # Get predictions
# lgb_preds = best_lgb.predict_proba(X_val)[:, 1]  # Probabilities for class 1  0.6296
# xgb_preds = best_xgb.predict_proba(X_val)[:, 1] #0.6267

# # Average predictions
# ensemble_preds = (lgb_preds + xgb_preds) / 2
# ensemble_preds = (ensemble_preds > 0.5).astype(int)  # Convert to binary labels

# # Evaluate ensemble model
# from sklearn.metrics import accuracy_score
# ensemble_acc = accuracy_score(y_val, ensemble_preds)

# print(f"Ensemble Validation Accuracy: {ensemble_acc:.4f}") #0.6272


from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(estimators=[
    ('lgb', best_lgb),
    ('xgb', best_xgb)
], voting='hard')  # 'soft' may also work

voting_clf.fit(X_train, y_train)
test_acc = voting_clf.score(X_val, y_val)
print(f"Ensemble Test Accuracy: {test_acc:.4f}")


In [ ]:
y_pred = voting_clf.predict(X_test_pre_df)
submission = pd.DataFrame({"id": range(0,len(y_pred)),
                           "target": y_pred})
submission.to_csv('submission.csv',index=False) 
print(submission) 

In [ ]:
st
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# Initialize GBM Classifier
gbm = GradientBoostingClassifier(random_state=42)

# Define hyperparameter grid
param_dist = {
    'n_estimators': [100, 200, 300],  # Number of boosting rounds  
    'learning_rate': [0.1, 0.05, 0.01],  # Step size shrinkage  
    'max_depth': [3, 5, 7],  # Tree depth  
    'subsample': [0.7, 0.8, 1.0],  # Prevent overfitting  
    'min_samples_split': [2, 5, 10],  # Minimum samples to split  
    'min_samples_leaf': [1, 2, 4]  # Minimum leaf nodes  
}

# Randomized Search with 20 iterations
random_search_gbm = RandomizedSearchCV(
    gbm, param_distributions=param_dist, n_iter=20, cv=3, scoring='accuracy', n_jobs=1, random_state=42
)

# Fit on training data
random_search_gbm.fit(X_train, y_train)

# Best GBM model
best_gbm = random_search_gbm.best_estimator_
gbm_val_accuracy = best_gbm.score(X_val, y_val)

print(f"Best Parameters for GBM: {random_search_gbm.best_params_}")
print(f"Validation Accuracy: {gbm_val_accuracy:.4f}")


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.1, 0.01, 0.001],
    'max_depth': [3, 5, 7, 9]
}

random_search_xgb = RandomizedSearchCV(XGBClassifier(random_state=42), param_dist, 
                                       n_iter=10, cv=3, scoring='accuracy', n_jobs=-1, random_state=42)

random_search_xgb.fit(X_train, y_train)
best_xgb_random = random_search_xgb.best_estimator_

print(f"Best Parameters: {random_search_xgb.best_params_}")
print(f"Validation Accuracy: {best_xgb_random.score(X_val, y_val):.4f}")

# Best Parameters Xgb randomized search: {'n_estimators': 50, 'max_depth': 5, 'learning_rate': 0.1}
# Validation Accuracy: 0.6195

# Model on PCA

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

solvers = ["lbfgs", "saga", "newton-cg", "liblinear"]
best_solver = None
best_score = 0

for solver in solvers:
    log_reg_cv = LogisticRegressionCV(cv=5, random_state=1, solver=solver, tol=1e-6, max_iter=1000)
    log_reg_cv.fit(X_pca_df, y_train)
    score = log_reg_cv.score(X_val_pca_df, y_val)
    
    print(f"Solver: {solver} | Validation Score: {score:.4f}")
    
    if score > best_score:
        best_score = score
        best_solver = solver

print(f"\n✅ Best Solver: {best_solver} with score {best_score:.4f}")

# Solver: lbfgs | Validation Score: 0.6036  handles l2 reg
# Solver: saga | Validation Score: 0.6036  uses sgd
# Solver: newton-cg | Validation Score: 0.6036  fast
# Solver: liblinear | Validation Score: 0.6035

# ✅ Best Solver: lbfgs with score 0.6036


In [ ]:
log_reg_cv = LogisticRegressionCV(cv=5, random_state=1, solver=best_solver, tol=1e-6, max_iter=1000)
log_reg_cv.fit(X_pca_df, y_train)

# Predict probabilities for the positive class
y_val_probs = log_reg_cv.predict_proba(X_val_pca_df)[:, 1]

from sklearn.metrics import roc_curve, auc #a graph that shows how well a model can distinguish between positive and negative examples

# Predict probabilities for the positive class
y_val_probs = log_reg_cv.predict_proba(X_val_pca_df)[:, 1]

# Compute ROC curve
fpr, tpr, _ = roc_curve(y_val, y_val_probs)
roc_auc = auc(fpr, tpr)

# Plot ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  # Diagonal reference line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show() 


print(f"ROC AUC Score: {roc_auc:.4f}") #0.64


In [ ]:
from sklearn.linear_model import RidgeCV

ridge_model = RidgeCV(alphas=np.logspace(-5, 5, 13)) 
ridge_model.fit(X_pca_df, y_train)
y_val_pred = ridge_model.predict(X_val_pca_df)

y_val_pred_binary = (y_val_pred >= 0.5).astype(int)
acc_score_ridge = accuracy_score(y_val, y_val_pred_binary)

print(acc_score_ridge) #0.6035 pca
ridge_model.alpha_ #316.227

print(ridge_model.alpha_)
print(np.logspace(-6, 6, 13))
(y_val_pred) #array

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import ShuffleSplit,GridSearchCV
from sklearn.metrics import classification_report

model_sgd = SGDClassifier(random_state=1, max_iter=10000, eta0 = 0.005, learning_rate='adaptive')
sgd = GridSearchCV(model_sgd,cv=ShuffleSplit(random_state=1),
                   param_grid = {
                       'eta0':[0.01,0.1],
                       'penalty':['l1','l2'],
                       'power_t':[0.01,2] #for inscaling
                   },verbose=2,return_train_score=True,
                   scoring = 'accuracy'
                  )
sgd.fit(X_pca_df,y_train)
y_val_pred = sgd.predict(X_val_pca_df)
print(classification_report(y_val, y_val_pred))

# score_sgd = accuracy_score(y_val, y_val_pred)
# print("SGD score:", score_sgd) #0.60

sgd.best_params_ #{'eta0': 0.1, 'penalty': 'l1', 'power_t': 0.01}

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

mlpclass = MLPClassifier(solver='adam', alpha=1e-5,
                    hidden_layer_sizes=(5, 5), random_state=1)
mlpclass.fit(X_pca_df, y_train)

print(classification_report(y_val, mlpclass.predict(X_val_pca_df))) 

#0.61 rfe, 0.60 pca, 0.58 df , submitted 0.603

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Define hyperparameter grid
param_grid = {
    'hidden_layer_sizes': [(50,), (50, 50), (100,)],  # More neurons
    'alpha': [1e-5, 1e-4, 1e-3],  # Regularization strength
    'learning_rate_init': [0.001, 0.01, 0.1],  # Learning rate tuning
}

# Initialize MLPClassifier
mlp = MLPClassifier(solver='adam', random_state=1, max_iter=500)

# Grid Search with cross-validation
grid_search = GridSearchCV(mlp, param_grid, cv=3, scoring='accuracy', n_jobs=1)
grid_search.fit(X_pca_df, y_train)

# Best model
best_mlp = grid_search.best_estimator_

# Evaluate on validation set
y_pred = best_mlp.predict(X_val_pca_df)
print("Best Parameters:", grid_search.best_params_)
print(classification_report(y_val, y_pred))


In [ ]:
clf1 = LogisticRegression()
clf2 = RandomForestClassifier()
clf3 = GradientBoostingClassifier()

voting_clf = VotingClassifier(
    estimators=[('lr', clf1), ('rf', clf2), ('gb', clf3)],
    voting='soft' #avg the prob
)
voting_clf.fit(X_pca_df, y_train)
voting_pred = voting_clf.predict(X_val_pca_df)
print(f"Voting Classifier Accuracy: {accuracy_score(y_val, voting_pred)}")
# 0.608 submitted

In [ ]:
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
param_grid = {
    'max_depth': [3, 5, 7], #more value, more overfit
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 200, 300],
    'min_child_weight': [1, 3, 5] #more val, less overfit
}

grid_search = GridSearchCV(xgb_model, param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_pca_df, y_train)
xgb_pred = grid_search.predict(X_val_pca_df)
print(f"XGBoost (tuned) Accuracy: {accuracy_score(y_val, xgb_pred)}")  #0.6078

In [ ]:
from lightgbm import LGBMClassifier

# Define the model
lgb = LGBMClassifier(random_state=42)

# Expanded hyperparameter grid
param_grid_lgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.1, 0.01, 0.05],
    'max_depth': [5, 7, 10],
    'num_leaves': [20, 31, 40],  # More flexibility in tree structure
    'min_child_samples': [10, 20, 30],  # Regularization
    'subsample': [0.8, 1.0],  # Helps generalization
    'colsample_bytree': [0.8, 1.0]  # Feature sampling
}

# Grid search with cross-validation
grid_search_lgb = GridSearchCV(lgb, param_grid_lgb, cv=5, scoring='accuracy', n_jobs=-1)
grid_search_lgb.fit(X_pca_df, y_train)

# Best model
best_lgb = grid_search_lgb.best_estimator_
lgb_val_accuracy = best_lgb.score(X_val_pca_df, y_val)

print(f"Best Parameters for LightGBM: {grid_search_lgb.best_params_}")
print(f"Validation Accuracy: {lgb_val_accuracy:.4f}") #time

# Best Parameters for XGBoost: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
# Validation Accuracy: 0.6221

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import GradientBoostingClassifier

# # Use the best-tuned models
# best_svc = grid_search_svc.best_estimator_
best_lgb = grid_search_lgb.best_estimator_
# best_lgb= LGBMClassifier(learning_rate=0.1, max_depth =5, n_estimators=100)
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3)

# Soft voting (requires probability=True for SVC)
voting_clf_est = VotingClassifier(
    estimators=[
        # ('svc', best_svc),
        ('lgbm', best_lgb),
        ('gb', gb)
    ],
    voting='soft'  # or 'hard'
)

voting_clf_est.fit(X_pca_df, y_train)

# y_pred = voting_clf_est.predict(X_test)
# print("Voting Classifier Accuracy:", accuracy_score(y_test, y_pred))


voting_clf_est_pred = voting_clf_est.predict(X_val_pca_df)
print(accuracy_score(y_val, voting_clf_est_pred)) #0.60865

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# each fold has the same class distribution 
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# Models to compare
models = {
    "Logistic Regression": LogisticRegression(random_state=1, max_iter=5000),#increased
    "Random Forest": RandomForestClassifier(random_state=1, n_estimators=100)
    }

# Evaluate models
best_model = None
best_score = 0

for name, model in models.items():
    score = cross_val_score(model, X_pca_df, y_train, cv=stratified_kfold, scoring='accuracy').mean()
    print(f"{name}: {score:.4f}")
    
    if score > best_score:
        best_score = score
        best_model = model

print(f"\nBest Model: {best_model} with CV accuracy: {best_score:.4f}")
# Logistic Regression: 0.6134
# Random Forest: 0.6128